# Multicriteria Model for Integrating Distributed Systems into Cloud Services

**Reproducibility notebook — runs end to end in Google Colab.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omega2417/bnt/blob/claude/zenodo-publication-project-sedznh/cloud-integration-mcdm/notebooks/Cloud_Integration_MCDM_Colab.ipynb)

This notebook reproduces every headline result of the article:

| Section | What is reproduced | Runtime |
|---|---|---|
| 1 | Setup and installation | ~30 s |
| 2 | The 18-system benchmark instance (Appendix A) | instant |
| 3 | The model: objectives, constraints, response curves | seconds |
| 4 | Exact enumeration of 262,144 portfolios | ~20 s |
| 5 | NSGA-II vs NSGA-III over matched seeds | 3-6 min (full) |
| 6 | Weighted-sum baseline | seconds |
| 7 | Paired Wilcoxon tests with Holm correction | instant |
| 8 | 25-cell temporal-rate sensitivity grid | ~6 min |
| 9 | Validation against the published numbers | instant |
| 10 | Export a results bundle | seconds |

**Runtime:** CPU is enough. No GPU, no accelerator, no Drive mount is required.

> The benchmark scenario is **synthetic**. It demonstrates that the model and the
> code behave as described; it is not evidence about any real organization.

---
## 1. Setup

Colab already ships compatible NumPy, SciPy, pandas and Matplotlib, so nothing
below forces a runtime restart. The cell clones the repository when running in
Colab and falls back to the local checkout otherwise, so the same notebook works
in both places.

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/omega2417/bnt.git"
BRANCH = "claude/zenodo-publication-project-sedznh"
PROJECT = "cloud-integration-mcdm"


def find_project_root(start: pathlib.Path) -> pathlib.Path:
    """Walk up from `start` until a directory containing src/cimcdm is found.

    This keeps the notebook working whether it is executed from the project
    root, from notebooks/, or from an unpacked Zenodo archive.
    """
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "cimcdm").is_dir():
            return candidate
        if (candidate / PROJECT / "src" / "cimcdm").is_dir():
            return candidate / PROJECT
    raise FileNotFoundError(
        f"Could not locate src/cimcdm above {start}. Run this notebook from "
        f"inside the {PROJECT} directory, or clone the repository first."
    )


already_present = any(
    (p / "src" / "cimcdm").is_dir()
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
)

if IN_COLAB and not already_present:
    if not pathlib.Path("bnt").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL],
            check=True,
        )
    root = pathlib.Path("bnt") / PROJECT
else:
    root = find_project_root(pathlib.Path.cwd().resolve())

sys.path.insert(0, str((root / "src").resolve()))
os.chdir(root)
print("project root:", root.resolve())

In [ ]:
# Only installs what is genuinely missing; Colab's preinstalled stack is fine.
try:
    import numpy, scipy, pandas, matplotlib  # noqa: F401
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
        check=True,
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cimcdm
from cimcdm import (
    AlgorithmConfig, DEFAULT_ALGORITHM, DEFAULT_SCENARIO,
    PortfolioModel, corner_summary, enumerate_exact, figures,
    load_published_instance, representative_run, run_benchmark,
    run_weighted_sum, sensitivity_grid,
)

%matplotlib inline
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print(f"cimcdm {cimcdm.__version__} | numpy {np.__version__} | pandas {pd.__version__}")

### Run size

`FULL_RUN = True` follows the published protocol exactly: 30 matched seeds and
the complete 25-cell sensitivity grid (roughly 10-12 minutes on a Colab CPU).
Set it to `False` for a 5-seed smoke pass that finishes in about a minute — the
exact-enumeration results are identical either way, only the evolutionary means
and the statistical tests become noisier.

In [ ]:
FULL_RUN = True

SEEDS = DEFAULT_ALGORITHM.seeds if FULL_RUN else DEFAULT_ALGORITHM.seeds[:5]
RUN_SENSITIVITY = FULL_RUN
print(f"{len(SEEDS)} matched seeds: {SEEDS[0]}-{SEEDS[-1]}")
print(f"sensitivity grid: {'25 cells' if RUN_SENSITIVITY else 'skipped'}")

---
## 2. The benchmark instance

Eighteen candidate distributed information-processing systems, with the exact
values printed in Appendix A of the article. Each system carries six temporal
parameters, four static attributes, an integration cost and an implementation
effort.

In [ ]:
instance = load_published_instance()
frame = instance.to_frame()
print(f"{instance.n} candidate systems, source: {instance.source}")
frame.round(6)

In [ ]:
print(f"total integration cost      {instance.cost.sum():8.1f} units")
print(f"total implementation effort {instance.effort.sum():8.1f} system-months")
print(f"total business criticality  {instance.criticality.sum():8.4f}")

---
## 3. The model

Three objectives, all minimized (Equation 1):

$$\min F(x,t) = \big[\,1 - \tilde V(x,t),\; \tilde C(x),\; \tilde R(x,t)\,\big],\qquad x \in \{0,1\}^{18}$$

Benefit, cost and risk are normalized so the three components are commensurate.
The normalization constant $V_{\max}$ is evaluated once at the **upper boundary**
of the tested temporal-rate range ($s_\alpha = s_\beta = 1.5$) and then held
fixed, which is what keeps objective values comparable across the entire
sensitivity grid in Section 8.

Temporal realization uses bounded monotone response functions (Equations 8-10):

$$P_i(t) = P_{i,0} + \Delta P_i\left[1 - e^{-\alpha_i t}\right],\quad
E_i(t) = E_{i,0} + \Delta E_i\left[1 - e^{-\beta_i t}\right],\quad
r_i(t) = r_{i,\infty} + (r_{i,0} - r_{i,\infty})e^{-\rho_i t}$$

In [ ]:
model = PortfolioModel(instance, DEFAULT_SCENARIO)

print(f"horizon t                      {model.scenario.horizon:.0f} months")
print(f"budget B                       {model.budget:.2f}   (58% of {model.total_cost:.0f})")
print(f"time cap Tmax                  {model.time_cap:.1f}    (62% of {model.total_effort:.0f})")
print(f"coverage minimum Kmin          {model.coverage_minimum:.4f} (46% of {model.total_criticality:.4f})")
print(f"min mean reliability qmin      {model.scenario.min_mean_reliability}")
print(f"min mean technical Tmin        {model.scenario.min_mean_technical}")
print(f"normalization constant Vmax    {model.v_max:.6f}")
print(f"hypervolume reference point    {model.scenario.reference_point}")

In [ ]:
fig = figures.figure_response_curves(model)
plt.show()

A dimensional sanity check from Section 2.2 of the article: with
$P_0 = 50$, $\Delta P = 20$, $\alpha = 1/3$ per month and $t = 3$ months, the
realized value is **62.6**, not 70 — the value 70 is the asymptote, reached only
as $t \to \infty$.

In [ ]:
P0, dP, alpha, t = 50.0, 20.0, 1/3, 3.0
realized = P0 + dP * (1 - np.exp(-alpha * t))
print(f"P(3)     = {realized:.1f}")
print(f"asymptote = {P0 + dP:.1f}")
assert abs(realized - 62.6) < 0.05

---
## 4. Exact enumeration

With 18 binary decisions there are only $2^{18} = 262{,}144$ candidate
portfolios, so the true Pareto front can be computed rather than approximated.
That front is the ground truth every heuristic below is scored against — which
is what separates *model validation* from *algorithm advocacy*.

In [ ]:
%time exact = enumerate_exact(model)

print(f"\nportfolios examined  {exact.n_total:,}")
print(f"feasible             {exact.n_feasible:,} ({100 * exact.feasible_fraction:.2f}%)")
print(f"Pareto-optimal       {exact.front_size}")
print(f"exact hypervolume    {exact.hypervolume:.9f}")
print(f"\npublished:           83,657 feasible | 446 front | 0.421542857")

### The knee portfolio

One illustrative compromise, chosen *after* the Pareto optimization by minimizing
the Euclidean distance to the ideal point under min-max scaling (Equation 11).
This is one preference model among many, not a claim that it is the right choice
for any particular organization.

In [ ]:
knee = exact.knee
print(f"selected ({len(knee.selected)} systems): {', '.join(knee.selected)}")
print(f"\nbenefit             {knee.benefit:.6f}")
print(f"normalized cost     {knee.objectives[1]:.6f}  ({knee.cost_units:.0f} cost units)")
print(f"normalized risk     {knee.objectives[2]:.6f}")
print(f"effort              {knee.effort_units:.0f} system-months (cap {model.time_cap:.1f})")
print(f"criticality         {knee.coverage:.6f} (min {model.coverage_minimum:.4f})")
print(f"mean reliability    {knee.mean_reliability:.6f} (min {model.scenario.min_mean_reliability})")
print(f"mean technical      {knee.mean_technical:.6f} (min {model.scenario.min_mean_technical})")

In [ ]:
front = pd.DataFrame(
    exact.F_front, columns=["f1_benefit_shortfall", "f2_cost", "f3_risk"]
)
front.insert(0, "systems_selected", exact.X_front.sum(axis=1).astype(int))
front.describe().round(6)

---
## 5. NSGA-II versus NSGA-III

Both methods share initialization, uniform crossover ($p = 0.90$), bit-flip
mutation ($p = 1/18$), binary tournament selection, the repair operator and the
generation budget. **They differ only in the survival step**: crowding distance
for NSGA-II, Das-Dennis reference directions for NSGA-III. Any measured
difference is therefore attributable to that one component.

This cell also runs the weighted-sum baseline and the paired statistical tests.

In [ ]:
config = AlgorithmConfig(seeds=SEEDS)
%time result = run_benchmark(instance, DEFAULT_SCENARIO, config)

In [ ]:
print("Table 5 - quality and runtime\n")
result.summary_table()

In [ ]:
print("Share of the exact hypervolume recovered:\n")
for method, percent in result.recovery_percentages().items():
    print(f"  {method:9s} {percent:6.2f}%")
print("\npublished: NSGA-II 99.31%, NSGA-III 99.39%, WSM 99.10%")

In [ ]:
print("Table 4 - convergence checkpoints\n")
result.convergence_table().round(6)

In [ ]:
fig = figures.figure_convergence(result.convergence, exact.hypervolume)
plt.show()

In [ ]:
fig = figures.figure_metric_distributions(result.runs)
plt.show()

---
## 6. Weighted-sum baseline

The weighted sum scores every feasible portfolio under 231 simplex-lattice weight
vectors and keeps the minimizers. It reaches almost the same **hypervolume** as
the evolutionary methods while recovering only a small fraction of the exact
front — the clearest demonstration in the study that a single quality indicator
can badly overstate a method's usefulness for exploratory decision support.

Its reported time covers scalar scoring of an already enumerated feasible matrix,
so it is **not** comparable with the evolutionary end-to-end runtimes.

In [ ]:
wsm_run, n_weights = run_weighted_sum(model, exact.F_feasible, exact.X_feasible, config)

print(f"weight vectors        {n_weights}")
print(f"solutions returned    {len(wsm_run.F)}")
print(f"hypervolume           {result.wsm['hypervolume']:.6f}  "
      f"({100 * result.wsm['hypervolume'] / exact.hypervolume:.2f}% of exact)")
print(f"exact-front coverage  {100 * result.wsm['coverage']:.2f}%")
print(f"\npublished: 29 solutions, HV 0.417759, coverage 6.50%")

In [ ]:
if result.run_objects:
    fig = figures.figure_front_projections(
        exact.F_front,
        representative_run(result, "NSGA-II").F,
        representative_run(result, "NSGA-III").F,
        wsm_run.F,
    )
    plt.show()

---
## 7. Paired statistical comparison

Two-sided Wilcoxon signed-rank tests on matched seeds, Pratt treatment of zero
differences, Holm correction across the five metrics. The rank-biserial effect
size is signed as NSGA-II minus NSGA-III, so its direction must be read together
with whether the metric is better-when-higher or better-when-lower.

With the full 30 seeds the article reports: no significant difference in
hypervolume or spacing, NSGA-III significantly better on IGD+ and coverage, and
NSGA-II significantly faster.

In [ ]:
print(f"Table 6 - paired Wilcoxon tests, Holm-corrected (n = {len(SEEDS)} seeds)\n")
result.tests_table().round(6)

---
## 8. Sensitivity to the temporal rates

Every adaptation rate $\alpha_i$ is multiplied by $s_\alpha$ and every
economic-benefit accumulation rate $\beta_i$ by $s_\beta$, each taking values in
$\{0.50, 0.75, 1.00, 1.25, 1.50\}$. **Exact enumeration is repeated in all 25
cells**, so the analysis can report whether the recommended portfolio itself
changes — not merely whether the indicator values move.

Takes roughly 6 minutes; skipped when `FULL_RUN = False`.

In [ ]:
if RUN_SENSITIVITY:
    %time grid = sensitivity_grid(instance, DEFAULT_SCENARIO, progress=True)
else:
    grid = None
    print("skipped (set FULL_RUN = True to run the 25-cell grid)")

In [ ]:
if grid is not None:
    display(corner_summary(grid).round(6))

    distinct = grid["knee_systems"].nunique()
    print(f"\ndistinct knee portfolios across {len(grid)} cells: {distinct}")
    if distinct == 1:
        print(f"invariant selection: {grid['knee_systems'].iloc[0]}")
        print("\nThe recommended portfolio is locally robust across the tested range.")
        print("This is evidence of local decision robustness, not of external validity.")

In [ ]:
if grid is not None:
    fig = figures.figure_sensitivity(grid)
    plt.show()

In [ ]:
if grid is not None:
    base = grid[(grid.s_alpha == 1.0) & (grid.s_beta == 1.0)].iloc[0]
    lo = grid[(grid.s_alpha == 0.5) & (grid.s_beta == 0.5)].iloc[0]
    hi = grid[(grid.s_alpha == 1.5) & (grid.s_beta == 1.5)].iloc[0]
    a_only = grid[(grid.s_beta == 1.0)]
    b_only = grid[(grid.s_alpha == 1.0)]

    pct = lambda new, old: 100 * (new - old) / old
    print(f"corner to corner, hypervolume  {pct(hi.hypervolume, lo.hypervolume):+.2f}%  (published +7.87%)")
    print(f"corner to corner, knee benefit {pct(hi.knee_benefit, lo.knee_benefit):+.2f}%  (published +8.04%)")
    print(f"s_alpha 0.5 -> 1.5 at s_beta=1  {pct(a_only.hypervolume.iloc[-1], a_only.hypervolume.iloc[0]):+.2f}%  (published +3.30%)")
    print(f"s_beta  0.5 -> 1.5 at s_alpha=1 {pct(b_only.hypervolume.iloc[-1], b_only.hypervolume.iloc[0]):+.2f}%  (published +4.34%)")

---
## 9. Validation against the published numbers

Each check re-derives one claim from the article and reports pass or fail with
the numeric residual. Residuals of order $10^{-7}$ are expected: Appendix A
publishes the inputs rounded to six decimal places, so the code cannot recover
more precision than the inputs carry.

The evolutionary checks are stochastic and use a looser tolerance; on a reduced
seed count they may legitimately drift outside it.

In [ ]:
from cimcdm.validation import report, validate_algorithms, validate_exact, validate_scenario, validate_sensitivity

ok = report(validate_scenario(model), "Scenario bounds (Table 2)")
ok &= report(validate_exact(exact), "Exact enumeration (Sections 3.1, 3.3)")
ok &= report(validate_algorithms(result), "Algorithm comparison (Table 5)")
if grid is not None:
    ok &= report(validate_sensitivity(grid), "Sensitivity grid (Table 7)")

print("\n" + ("ALL CHECKS PASSED" if ok else "SOME CHECKS FAILED - see above"))

### Comparison with the published run-level tables

Appendix B lists the article's own 30 run-level outcomes. Our runs use the same
seeds but a re-implemented variation pipeline, so individual runs will not match
value for value; what should agree is the **distribution** and the direction of
the differences between the two methods.

In [ ]:
published_b1 = pd.read_csv("data/appendix_B1_run_quality.csv")
published_b2 = pd.read_csv("data/appendix_B2_run_coverage_time.csv")

comparison = pd.DataFrame({
    "metric": ["HV NSGA-II", "HV NSGA-III", "IGD+ NSGA-II", "IGD+ NSGA-III",
               "spacing NSGA-II", "spacing NSGA-III"],
    "published_mean": [
        published_b1.hv_ii.mean(), published_b1.hv_iii.mean(),
        published_b1.igdp_ii.mean(), published_b1.igdp_iii.mean(),
        published_b1.sp_ii.mean(), published_b1.sp_iii.mean(),
    ],
    "this_run_mean": [
        result.runs.query("method == 'NSGA-II'").hypervolume.mean(),
        result.runs.query("method == 'NSGA-III'").hypervolume.mean(),
        result.runs.query("method == 'NSGA-II'").igd_plus.mean(),
        result.runs.query("method == 'NSGA-III'").igd_plus.mean(),
        result.runs.query("method == 'NSGA-II'").spacing.mean(),
        result.runs.query("method == 'NSGA-III'").spacing.mean(),
    ],
})
comparison["difference"] = comparison.this_run_mean - comparison.published_mean
comparison.round(6)

---
## 10. Export the results bundle

Writes every table and figure to `results/`. In Colab the next cell zips the
folder and downloads it, so the outputs survive the runtime being recycled.

In [ ]:
import pathlib

out = pathlib.Path("results")
(out / "figures").mkdir(parents=True, exist_ok=True)

result.runs.to_csv(out / "run_level_metrics.csv", index=False)
result.convergence.to_csv(out / "convergence_by_generation.csv", index=False)
result.tests_table().to_csv(out / "wilcoxon_holm_tests.csv", index=False)
result.summary_table().to_csv(out / "quality_summary.csv", index=False)
front.to_csv(out / "exact_front_objectives.csv", index=False)
pd.DataFrame(exact.X_front.astype(int), columns=list(instance.names)).to_csv(
    out / "exact_front_portfolios.csv", index=False
)
if grid is not None:
    grid.to_csv(out / "sensitivity_grid.csv", index=False)

figures.figure_convergence(result.convergence, exact.hypervolume, out / "figures/figure2_convergence.png")
figures.figure_metric_distributions(result.runs, out / "figures/figure3_distributions.png")
if result.run_objects:
    figures.figure_front_projections(
        exact.F_front,
        representative_run(result, "NSGA-II").F,
        representative_run(result, "NSGA-III").F,
        wsm_run.F,
        out / "figures/figure4_projections.png",
    )
if grid is not None:
    figures.figure_sensitivity(grid, out / "figures/figure5_sensitivity.png")
figures.figure_response_curves(model, out / "figures/figureA_response_curves.png")
plt.close("all")

for path in sorted(out.rglob("*")):
    if path.is_file():
        print(f"  {path}  ({path.stat().st_size:,} bytes)")

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files

    shutil.make_archive("cimcdm_results", "zip", "results")
    files.download("cimcdm_results.zip")
else:
    print("Not in Colab; results are on disk under results/")

---
## Scope and limitations

Reproducing these numbers establishes internal computational correctness. It does
**not** establish that the model works in a deployed organization. In particular:

- The trading-company scenario is synthetic; no real organizational data was used.
- Decisions are binary. Partial migration, hybrid-cloud allocation, sequencing,
  rollback and multi-cloud placement are outside the decision space.
- System contributions are additive. Interdependencies, shared services and
  correlated failures would change the Pareto set.
- The rates $\alpha$, $\beta$ and $\rho$ are not empirically calibrated; the
  25-cell grid is a bounded sensitivity exercise, not uncertainty quantification.
- The reliability and technical-readiness thresholds constrain portfolio
  *averages*, not each selected system individually.
- Algorithm evidence covers one 18-system instance, one parameterization and one
  matched budget, with no parameter tuning.

### Citation

> Torstensson, O.; Prokopovych-Tkachenko, D.; Lakhno, V.; Desiatko, A.;
> Fedotov, S. *Multicriteria Model for Integrating Distributed Systems into
> Cloud Services.* Systems, 2026 (submitted manuscript).

Software licensed under MIT; the transcribed appendix data under CC BY 4.0.